# Geneformer V2-316M 字典文件概览

本 notebook 只读检查 Geneformer V2-316M 使用的三个字典文件，不执行模型部署，也不修改原始数据：

- `token_dictionary_gc104M.pkl`：模型实际词表，映射规范 Ensembl gene ID 到 token ID。
- `gene_name_id_dict_gc104M.pkl`：基因名称或 ID 到规范 Ensembl gene ID 的辅助映射。
- `ensembl_mapping_dict_gc104M.pkl`：别名、旧 ID 或规范 ID 到规范 Ensembl gene ID 的归并映射。

> Pickle 文件可能包含可执行对象。本 notebook 会先核对官方文件的 SHA-256，并使用禁止加载全局对象的受限 Unpickler；仍然只应检查可信来源的文件。

In [1]:
from collections import Counter
from hashlib import sha256
from pathlib import Path
import pickle
import re

import pandas as pd
from IPython.display import display


pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 140)

project_root = Path("/home/svu/e1538713/CodeNo0")
if not project_root.exists():
    project_root = Path("/nfs/home/svu/e1538713/CodeNo0")

data_dir = project_root / "data" / "raw" / "Geneformer"
files = {
    "token_dictionary": data_dir / "token_dictionary_gc104M.pkl",
    "gene_name_id_dict": data_dir / "gene_name_id_dict_gc104M.pkl",
    "ensembl_mapping_dict": data_dir / "ensembl_mapping_dict_gc104M.pkl",
}
expected_sha256 = {
    "token_dictionary": "67c445f4385127adfc48dcc072320cd65d6822829bf27dd38070e6e787bc597f",
    "gene_name_id_dict": "fabfa0c2f49c598c59ae432a32c3499a5908c033756c663b5e0cddf58deea8e1",
    "ensembl_mapping_dict": "0819bcbd869cfa14279449b037eb9ed1d09a91310e77bd1a19d927465030e95c",
}

print(f"项目根目录: {project_root.resolve()}")
print(f"字典目录: {data_dir.resolve()}")

项目根目录: /nfs/home/svu/e1538713/CodeNo0
字典目录: /scratch/e1538713/CodeNo0/data/raw/Geneformer


## 1. 文件元数据与完整性校验

In [2]:
def file_sha256(path, chunk_size=1024 * 1024):
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


file_rows = []
for name, path in files.items():
    exists = path.is_file()
    actual_hash = file_sha256(path) if exists else None
    file_rows.append(
        {
            "name": name,
            "file": path.name,
            "exists": exists,
            "size_MiB": round(path.stat().st_size / 1024**2, 3) if exists else None,
            "sha256": actual_hash,
            "matches_official_sha256": actual_hash == expected_sha256[name],
        }
    )

file_metadata = pd.DataFrame(file_rows).set_index("name")
display(file_metadata)

if not file_metadata["exists"].all():
    missing = file_metadata.index[~file_metadata["exists"]].tolist()
    raise FileNotFoundError(f"缺少字典文件: {missing}")

if not file_metadata["matches_official_sha256"].all():
    mismatched = file_metadata.index[~file_metadata["matches_official_sha256"]].tolist()
    raise ValueError(f"以下文件与已核实的官方 SHA-256 不一致，停止反序列化: {mismatched}")

print("三个文件均存在，并且 SHA-256 与 Geneformer 官方文件一致。")

,file,exists,size_MiB,sha256,matches_official_sha256
name,,,,,
token_dictionary,token_dictionary_gc104M.pkl,True,0.406,67c445f4385127adfc48dcc072320cd65d6822829bf27dd38070e6e787bc597f,True
gene_name_id_dict,gene_name_id_dict_gc104M.pkl,True,1.584,fabfa0c2f49c598c59ae432a32c3499a5908c033756c663b5e0cddf58deea8e1,True
ensembl_mapping_dict,ensembl_mapping_dict_gc104M.pkl,True,3.774,0819bcbd869cfa14279449b037eb9ed1d09a91310e77bd1a19d927465030e95c,True


三个文件均存在，并且 SHA-256 与 Geneformer 官方文件一致。


## 2. 安全加载与整体结构

In [3]:
class RestrictedUnpickler(pickle.Unpickler):
    """拒绝 pickle 导入或构造任意全局对象。"""

    def find_class(self, module, name):
        raise pickle.UnpicklingError(
            f"不允许加载全局对象: {module}.{name}"
        )


def load_plain_dictionary(path):
    with path.open("rb") as handle:
        obj = RestrictedUnpickler(handle).load()
    if not isinstance(obj, dict):
        raise TypeError(f"{path.name} 的顶层对象不是 dict，而是 {type(obj).__name__}")
    return obj


dictionaries = {name: load_plain_dictionary(path) for name, path in files.items()}

def type_counts(values):
    counts = Counter(type(value).__name__ for value in values)
    return ", ".join(f"{type_name}: {count:,}" for type_name, count in counts.items())


structure_rows = []
for name, obj in dictionaries.items():
    structure_rows.append(
        {
            "name": name,
            "python_type": type(obj).__name__,
            "entries": len(obj),
            "key_types": type_counts(obj.keys()),
            "value_types": type_counts(obj.values()),
        }
    )

structure_summary = pd.DataFrame(structure_rows).set_index("name")
structure_summary["entries"] = structure_summary["entries"].map(lambda value: f"{value:,}")
display(structure_summary)

,python_type,entries,key_types,value_types
name,,,,
token_dictionary,dict,"20,275","str: 20,275","int: 20,275"
gene_name_id_dict,dict,"63,675","str: 63,675","str: 63,675"
ensembl_mapping_dict,dict,"173,697","str: 173,697","str: 173,697"


## 3. `token_dictionary_gc104M.pkl`

这是 V2-316M 输入嵌入矩阵的实际行索引。数字 token ID 必须与模型 embedding 的行号保持一致。

In [4]:
token_dictionary = dictionaries["token_dictionary"]
special_tokens = {key: value for key, value in token_dictionary.items() if key.startswith("<")}
gene_token_items = [
    (ensembl_id, token_id)
    for ensembl_id, token_id in token_dictionary.items()
    if ensembl_id not in special_tokens
]
gene_token_ids = [token_id for _, token_id in gene_token_items]
canonical_ensembl_pattern = re.compile(r"^ENSG\d{11}$")
versioned_ensembl_pattern = re.compile(r"^ENSG\d{11}\.\d+$")

token_checks = pd.Series(
    {
        "总 token 数": len(token_dictionary),
        "基因 token 数": len(gene_token_items),
        "特殊 token 数": len(special_tokens),
        "最小 token ID": min(token_dictionary.values()),
        "最大 token ID": max(token_dictionary.values()),
        "token ID 是否唯一": len(set(token_dictionary.values())) == len(token_dictionary),
        "token ID 是否连续": set(token_dictionary.values()) == set(range(len(token_dictionary))),
        "规范 ENSG 基因 ID 数": sum(bool(canonical_ensembl_pattern.fullmatch(key)) for key, _ in gene_token_items),
        "含版本后缀的基因 ID 数": sum(bool(versioned_ensembl_pattern.fullmatch(key)) for key, _ in gene_token_items),
        "是否存在 <unk>": "<unk>" in token_dictionary,
    },
    name="value",
)
display(token_checks.to_frame())

print("特殊 token：")
display(pd.DataFrame(special_tokens.items(), columns=["token", "token_id"]))

token_df = pd.DataFrame(gene_token_items, columns=["ensembl_id", "token_id"]).sort_values("token_id")
print("前 10 个基因 token：")
display(token_df.head(10))
print("最后 10 个基因 token：")
display(token_df.tail(10))

,value
总 token 数,20275
基因 token 数,20271
特殊 token 数,4
最小 token ID,0
最大 token ID,20274
token ID 是否唯一,True
token ID 是否连续,True
规范 ENSG 基因 ID 数,20271
含版本后缀的基因 ID 数,0
是否存在 <unk>,False


特殊 token：


,token,token_id
0,<pad>,0
1,<mask>,1
2,<cls>,2
3,<eos>,3


前 10 个基因 token：


,ensembl_id,token_id
0,ENSG00000000003,4
1,ENSG00000000005,5
2,ENSG00000000419,6
3,ENSG00000000457,7
4,ENSG00000000460,8
5,ENSG00000000938,9
6,ENSG00000000971,10
7,ENSG00000001036,11
8,ENSG00000001084,12
9,ENSG00000001167,13


最后 10 个基因 token：


,ensembl_id,token_id
20261,ENSG00000292277,20265
20262,ENSG00000292320,20266
20263,ENSG00000292323,20267
20264,ENSG00000292324,20268
20265,ENSG00000292326,20269
20266,ENSG00000292379,20270
20267,ENSG00000292438,20271
20268,ENSG00000293543,20272
20269,ENSG00000293552,20273
20270,ENSG00000293553,20274


## 4. `gene_name_id_dict_gc104M.pkl`

该文件比模型词表更宽，不能直接视为 Geneformer 的基因词表；必须再检查其目标 Ensembl ID 是否存在于 `token_dictionary`。

In [5]:
gene_name_id_dict = dictionaries["gene_name_id_dict"]
v2_gene_ids = set(token_df["ensembl_id"])
gene_name_targets = set(gene_name_id_dict.values())

gene_name_summary = pd.Series(
    {
        "总映射数": len(gene_name_id_dict),
        "唯一目标 Ensembl ID 数": len(gene_name_targets),
        "目标位于 V2 词表的映射数": sum(value in v2_gene_ids for value in gene_name_id_dict.values()),
        "目标不在 V2 词表的映射数": sum(value not in v2_gene_ids for value in gene_name_id_dict.values()),
        "key 本身为规范 ENSG 的映射数": sum(bool(canonical_ensembl_pattern.fullmatch(key)) for key in gene_name_id_dict),
        "V2 基因是否全部被该文件覆盖": v2_gene_ids.issubset(gene_name_targets),
    },
    name="value",
)
display(gene_name_summary.to_frame())

gene_name_df = pd.DataFrame(
    gene_name_id_dict.items(),
    columns=["gene_name_or_id", "canonical_ensembl_id"],
)
print("前 10 条映射：")
display(gene_name_df.head(10))

,value
总映射数,63675
唯一目标 Ensembl ID 数,63675
目标位于 V2 词表的映射数,20271
目标不在 V2 词表的映射数,43404
key 本身为规范 ENSG 的映射数,22601
V2 基因是否全部被该文件覆盖,True


前 10 条映射：


,gene_name_or_id,canonical_ensembl_id
0,5S_rRNA,ENSG00000277411
1,5_8S_rRNA,ENSG00000277739
2,7SK,ENSG00000271394
3,A1BG,ENSG00000121410
4,A1BG-AS1,ENSG00000268895
5,A1CF,ENSG00000148584
6,A2M,ENSG00000175899
7,A2M-AS1,ENSG00000245105
8,A2ML1,ENSG00000166535
9,A2ML1-AS1,ENSG00000256661


## 5. `ensembl_mapping_dict_gc104M.pkl`

该文件用于将规范 ID、旧 ID 和名称/别名归并到规范 Ensembl ID。多个输入 key 可能指向同一个目标，因此它不是一一映射。

In [6]:
ensembl_mapping_dict = dictionaries["ensembl_mapping_dict"]
mapping_targets = set(ensembl_mapping_dict.values())

mapping_summary = pd.Series(
    {
        "总映射数": len(ensembl_mapping_dict),
        "唯一目标 Ensembl ID 数": len(mapping_targets),
        "identity 映射数": sum(key == value for key, value in ensembl_mapping_dict.items()),
        "ENSG 格式 source 数": sum(bool(canonical_ensembl_pattern.fullmatch(key)) for key in ensembl_mapping_dict),
        "名称或别名 source 数": sum(not bool(canonical_ensembl_pattern.fullmatch(key)) for key in ensembl_mapping_dict),
        "目标位于 V2 词表的映射数": sum(value in v2_gene_ids for value in ensembl_mapping_dict.values()),
        "唯一 V2 目标数": len(mapping_targets & v2_gene_ids),
        "含版本后缀的 Ensembl source 数": sum(bool(versioned_ensembl_pattern.fullmatch(key)) for key in ensembl_mapping_dict),
        "目标集合是否等于 gene_name_id_dict 目标集合": mapping_targets == gene_name_targets,
    },
    name="value",
)
display(mapping_summary.to_frame())

mapping_df = pd.DataFrame(
    ensembl_mapping_dict.items(),
    columns=["source_name_or_id", "canonical_ensembl_id"],
)
print("前 10 条映射：")
display(mapping_df.head(10))

alias_examples = [alias for alias in ["P53", "HER1", "ERBB1"] if alias in ensembl_mapping_dict]
alias_example_df = pd.DataFrame(
    [
        {
            "alias": alias,
            "canonical_ensembl_id": ensembl_mapping_dict[alias],
            "geneformer_token_id": token_dictionary.get(ensembl_mapping_dict[alias]),
        }
        for alias in alias_examples
    ]
)
print("名称/别名归并示例：")
display(alias_example_df)

,value
总映射数,173697
唯一目标 Ensembl ID 数,63675
identity 映射数,63675
ENSG 格式 source 数,71783
名称或别名 source 数,101914
目标位于 V2 词表的映射数,91753
唯一 V2 目标数,20271
含版本后缀的 Ensembl source 数,0
目标集合是否等于 gene_name_id_dict 目标集合,True


前 10 条映射：


,source_name_or_id,canonical_ensembl_id
0,5S_RRNA,ENSG00000277411
1,RNA5-8SP10,ENSG00000277739
2,5_8S_RRNA,ENSG00000277739
3,7SK,ENSG00000271394
4,ABG,ENSG00000121410
5,GAB,ENSG00000121410
6,HYST2477,ENSG00000121410
7,A1BG,ENSG00000121410
8,A1BG-AS1,ENSG00000268895
9,NCRNA00181,ENSG00000268895


名称/别名归并示例：


,alias,canonical_ensembl_id,geneformer_token_id
0,P53,ENSG00000141510,8014
1,HER1,ENSG00000146648,8744
2,ERBB1,ENSG00000146648,8744


## 6. 从常用 gene symbol 到 Geneformer token 的联表示例

In [7]:
example_symbols = ["TP53", "BRCA1", "EGFR", "MYC", "GAPDH"]
crosswalk_examples = []
for symbol in example_symbols:
    ensembl_id = gene_name_id_dict.get(symbol)
    crosswalk_examples.append(
        {
            "gene_symbol": symbol,
            "canonical_ensembl_id": ensembl_id,
            "geneformer_token_id": token_dictionary.get(ensembl_id),
            "in_v2_vocabulary": ensembl_id in v2_gene_ids,
        }
    )

display(pd.DataFrame(crosswalk_examples))

cross_checks = pd.Series(
    {
        "20,271 个 V2 基因均有 gene_name_id_dict 目标": v2_gene_ids.issubset(gene_name_targets),
        "20,271 个 V2 基因均有 ensembl_mapping_dict 目标": v2_gene_ids.issubset(mapping_targets),
        "两个辅助字典的目标集合完全相同": gene_name_targets == mapping_targets,
    },
    name="通过",
)
display(cross_checks.to_frame())

,gene_symbol,canonical_ensembl_id,geneformer_token_id,in_v2_vocabulary
0,TP53,ENSG00000141510,8014,True
1,BRCA1,ENSG00000012048,312,True
2,EGFR,ENSG00000146648,8744,True
3,MYC,ENSG00000136997,7286,True
4,GAPDH,ENSG00000111640,3880,True


,通过
"20,271 个 V2 基因均有 gene_name_id_dict 目标",True
"20,271 个 V2 基因均有 ensembl_mapping_dict 目标",True
两个辅助字典的目标集合完全相同,True


## 结论

- V2-316M 的实际词表共有 **20,275** 个 token，其中 **20,271** 个是基因，另外 4 个是特殊 token。
- 三个字典中的规范基因标识均是不带 `.版本号` 的 Ensembl gene ID。
- `gene_name_id_dict` 和 `ensembl_mapping_dict` 覆盖的基因范围大于模型词表；后续部署时必须以 `token_dictionary` 是否包含目标 Ensembl ID 作为最终判断。
- `ensembl_mapping_dict` 允许多个名称或旧 ID 归并到同一基因，因此正式构建一一对应表时需要单独报告重复归并和未匹配基因。
- 本 notebook 到此只完成文件内容检查，未提取模型 embedding，也未生成或覆盖任何部署文件。